# Apache Iceberg - Python

All 14 Python examples from [docs/iceberg.md](https://platob.github.io/yggdryl/iceberg/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")

# A table is created in a folder, and a folder is all it ever touches.
table = Table.create(root, schema, ["venue"])

# A table that has never been written to has no current snapshot.
assert table.current_snapshot is None
assert table.scan().read_all().num_rows == 0

table.append(
    pa.record_batch(
        {"id": [1, 2], "venue": ["XNAS", "XNYS"]},
        schema=pa.schema([
            pa.field("id", pa.int64(), nullable=False),
            pa.field("venue", pa.string()),
        ]),
    )
)

assert table.current_snapshot is not None
assert table.current_snapshot.operation == "append"
assert len(table.data_files()) == 2, "one file per venue"

# Reopening finds the table again, with no catalog in between.
reopened = Table.open(IOBase(root.url.to_path()))
assert reopened.scan().read_all().num_rows == 2

## What a table writes

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")

table = Table.create(root, schema)
table.append(
    pa.record_batch(
        {"id": [1]}, schema=pa.schema([pa.field("id", pa.int64(), nullable=False)])
    )
)

# `table.root` is the folder handle the table reads and writes through.
names = [
    entry.name
    for entry in table.root.ls(recursive=True)
    if entry.is_file()
]

# One Parquet data file, one manifest, one manifest list, two metadata
# documents (create, then commit), and the version hint that finds them.
assert any(name.endswith(".parquet") for name in names)
assert any(name.startswith("snap-") and name.endswith(".avro") for name in names)
assert any(name.endswith("-m0.avro") for name in names)
assert "v1.metadata.json" in names
assert "v2.metadata.json" in names
assert "version-hint.text" in names

## Manifest lists and manifests

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = columns

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema, ["venue"])
table.append(
    pa.record_batch({"id": [1, 2], "venue": ["XNAS", "XNAS"]}, schema=columns)
)

# A snapshot names one manifest list; each of its rows is a manifest.
manifests = table.manifests()
assert len(manifests) == 1
assert manifests[0].is_data()
assert manifests[0].added_files_count == 1
assert manifests[0].added_rows_count == 2

# Each manifest row is a data file plus what the writer measured about it.
(file, spec), = table.data_files()
assert file.file_format == "PARQUET"
assert file.record_count == 2
assert spec.fields[0].name == "venue"

# Statistics are keyed by field id, which is what lets a planner skip a file.
assert file.value_counts[1] == 2
assert 1 in file.column_sizes

## Partition specs and the Hive layout

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = columns

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema, ["venue"])
table.append(pa.record_batch({"id": [1, 2], "venue": ["XNAS", None]}, schema=columns))

files = table.data_files()
assert len(files) == 2
null_file, _ = next(pair for pair in files if pair[0].partition[0] is None)
assert "venue=null" in null_file.path, "the path spells it"
assert null_file.partition[0] is None, "the manifest means it"

## Reading with column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
schema = columns

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema)
table.append(
    pa.record_batch({"id": [1, 2], "symbol": ["AAPL", "MSFT"]}, schema=columns)
)

# The target names the columns to keep; each file's Parquet reader gets it as
# its own projection mask, so the dropped column chunk is never decoded.
wanted = pa.schema([pa.field("id", pa.int64(), nullable=False)])
reader = table.scan(wanted)
assert reader.schema.names == ["id"]
assert reader.read_all().num_rows == 2

# No target reads everything.
assert table.scan().schema.names == ["id", "symbol"]

## Time travel and the inspection tables

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])
root = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-")) / "trades"

table = Table.create(IOBase(root), columns)
table.append(pa.record_batch({"id": [1]}, schema=columns))
past = table.current_snapshot.snapshot_id
table.overwrite(pa.record_batch({"id": [9]}, schema=columns))

# The present shows the overwrite; the retained snapshot shows what was.
assert table.scan().read_all().column("id").to_pylist() == [9]
assert table.scan_at(past).read_all().column("id").to_pylist() == [1]

# A branch or tag resolves by name, and every commit moves `main`.
assert table.snapshot_by_ref("main").snapshot_id == table.current_snapshot.snapshot_id

# The inspection readers render the table's own record as record batches.
assert table.inspect_history().read_all().num_rows == 2
assert table.inspect_snapshots().read_all().column("operation").to_pylist() == [
    "append",
    "overwrite",
]
assert table.inspect_files().read_all().num_rows == 1

shutil.rmtree(root.parent)

## The three record methods over a table

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("venue", pa.string()),
])
schema = columns
rows = lambda ids, venues: pa.record_batch(
    {"id": ids, "venue": venues}, schema=columns
)

path = pathlib.Path(tempfile.mkdtemp()) / "trades"
Table.create(IOBase(path), schema, ["venue"])

# The folder *is* the table, so the ordinary record surface reaches it. Its
# options come from the metadata, before a single data file exists.
folder = IOBase(path)
options = folder.record_options()
folder.write_arrow_batch_reader(rows([1, 2], ["XNAS", "XNYS"]), options=options)
folder.append_arrow_batch_reader(rows([3], ["XLON"]), options=options)

# A match key upserts: `2` is stored and updates, `9` is new and appends.
merging = folder.record_options()
merging.merge_by_names = ["id"]
folder.write_arrow_batch_reader(rows([2, 9], ["XNYS", "XLON"]), options=merging)

assert folder.read_arrow_batch_reader(options=options).read_all().num_rows == 4

# Each call was one commit, and the read went through the last one.
assert len(Table.open(IOBase(path)).snapshots) == 3

## A warehouse of tables

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl import DataType, Field
from yggdryl.iceberg import Catalog

warehouse = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-")) / "warehouse"
catalog = Catalog(warehouse)

# Rows and a name are enough: the first append creates the table with the
# schema the rows carry, and the second appends to it.
marked = Field(
    "row",
    DataType.from_fields([
        Field("id", "int64", nullable=False),
        Field("venue", "string"),
    ]),
    nullable=False,
).with_partition_fields(["venue"])
columns = pa.schema([child.to_arrow() for child in marked.data_type])

table = catalog.append(
    "nyc.trades", pa.table({"id": [1, 2], "venue": ["XNAS", "XNYS"]}, schema=columns)
)
assert table.scan().read_all().num_rows == 2

catalog.append("nyc.trades", pa.table({"id": [3], "venue": ["XNAS"]}, schema=columns))

# The partition marks the schema carried became the table's spec.
reopened = catalog.table("nyc.trades")
assert [field.name for field in reopened.spec.fields] == ["venue"]
assert reopened.scan().read_all().num_rows == 3
assert catalog.has_table("nyc.trades")
assert catalog.list_namespaces() == ["nyc"]
assert catalog.list_tables("nyc") == ["nyc.trades"]

shutil.rmtree(warehouse.parent)

## Data files aim at a size

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl.iceberg import Catalog

warehouse = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-")) / "warehouse"
catalog = Catalog(warehouse)

# The default target is Iceberg's own 512 MiB.
columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])
table = catalog.create_table("tiny.rows", columns)
assert table.target_file_size == 512 * 1024 * 1024

# Five appends, five snapshots, five small files.
for value in range(5):
    table.append(pa.record_batch({"id": [value]}, schema=columns))
assert table.inspect_files().read_all().num_rows == 5

# Compaction rewrites the small groups as one replace commit and reports it.
compaction = table.compact()
assert compaction.files_before == 5
assert compaction.files_after == 1
assert compaction.bytes_rewritten > 0
assert table.scan().read_all().num_rows == 5

# Nothing to do is a no-op that commits nothing.
done = table.compact()
assert (done.files_before, done.files_after, done.bytes_rewritten) == (0, 0, 0)

shutil.rmtree(warehouse.parent)

## Schema evolution and field ids

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])
schema = columns

root = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades")
table = Table.create(root, schema)
table.append(pa.record_batch({"id": [1]}, schema=columns))

# Add a column. Numbering continues above `last-column-id`, so the new column
# can never be confused with a dropped one.
evolved = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("quantity", pa.int64()),
])
assert table.evolve_schema(evolved) == 1, "the new schema's id"

# The old schema is retained, so the snapshot written under it still reads.
assert len(table.schemas) == 2
assert len(table.schemas[0].data_type) == 1

# And the file written before the column existed reads it as null.
rows = table.scan().read_all()
assert rows.column_names == ["id", "quantity"]
assert rows.column("quantity").null_count == rows.num_rows

In [ ]:
import pyarrow as pa

from yggdryl.iceberg import assign_field_ids

columns = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field(
        "leg",
        pa.struct([pa.field("price", pa.decimal128(18, 4), nullable=False)]),
    ),
])

# Depth first from `start`; the numbered schema is what comes back, so the
# schema handed in is left as it was.
schema = assign_field_ids(columns, 1)
assert [child.parquet_field_id for child in schema.data_type] == [1, 2]
assert schema.data_type[1].data_type[0].parquet_field_id == 3

# The root is not a column, so it is not numbered.
assert schema.parquet_field_id is None

# A field that already carries an id keeps it, so a second pass changes nothing.
assert [child.parquet_field_id for child in assign_field_ids(schema, 100).data_type] == [1, 2]

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase
from yggdryl.iceberg import Table

# A plain PyArrow schema carries no ids; creating the table numbers it.
columns = pa.schema([pa.field("id", pa.int64(), nullable=False)])
table = Table.create(IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades"), columns)

assert [child.parquet_field_id for child in table.schema.data_type] == [1]

## Evolving a schema

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa
import pytest

from yggdryl import IOBase
from yggdryl.iceberg import Table, can_promote

# Legal promotions pass; anything else is refused naming both sides.
assert can_promote("int32", "int64") is None
assert can_promote("decimal128(10, 2)", "decimal128(18, 2)") is None
with pytest.raises(ValueError, match="int64 to int32"):
    can_promote("int64", "int32")

columns = pa.schema([
    pa.field("id", pa.int32(), nullable=False),
    pa.field("symbol", pa.string()),
])
root = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-")) / "trades"
table = Table.create(IOBase(root), columns)

# Widen id, rename symbol, add venue - one evolved schema, one commit.
with table.update_schema() as update:
    update.update_type("id", "int64").rename_column("symbol", "ticker")
    update.add_column("", "venue: string")

children = list(table.schema.data_type)
assert [child.name for child in children] == ["id", "ticker", "venue"]
assert str(children[0].data_type) == "int64"
# A renamed column keeps its identifier: the name is a label, the id is the column.
assert [child.parquet_field_id for child in children] == [1, 2, 3]

shutil.rmtree(root.parent)

## Schemas as documents

In [ ]:
import json

from yggdryl.iceberg import schema_from_json, schema_to_json

document = json.loads("""{"type":"struct","schema-id":0,"fields":[
    {"id":1,"name":"id","required":true,"type":"long"},
    {"id":2,"name":"symbol","required":false,"type":"string"}
]}""")

# An Iceberg schema is a non-null struct field; its columns are the children.
schema = schema_from_json("row", document)
assert schema.data_type.kind == "struct"
assert not schema.nullable
assert len(schema.data_type) == 2
assert str(schema.data_type[0].data_type) == "int64"

# `required` inverts into nullability, and `id` becomes PARQUET:field_id.
assert not schema.data_type[0].nullable
assert schema.data_type[1].nullable
assert schema.data_type[0].parquet_field_id == 1
assert schema.data_type[0]["PARQUET:field_id"] == "1"

# The same document comes back out.
assert schema_to_json(schema) == document